In [1]:
%load_ext dotenv
%dotenv

In [2]:
import pandas as pd
import pinecone
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer
import hashlib

## Data Loading and preprocessing

In [4]:
courses = pd.read_csv('course_section_descriptions.csv', encoding='ANSI')

In [9]:
courses.head()

,course_id,course_name,course_slug,course_description,course_description_short,course_technology,course_topic,course_instructor_quote,section_id,section_name,section_description,unique_id
0,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,9,Introduction to Tableau,While Tableau is an indispensable tool in the ...,2_9
1,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,10,Tableau Functionalities,"In this section, you will create your first Ta...",2_10
2,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,11,The Tableau Exercise,This section is a practical example that will ...,2_11
3,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,12,Introduction,"In this section, you will learn about the impo...",3_12
4,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,13,Setting Up the Environments,"Here, we set up different environments for the...",3_13


In [7]:
courses["unique_id"] = courses["course_id"].astype(str) + "_" + courses["section_id"].astype(str)

In [14]:
def create_section_description(row):
    return f"""
        The course name is {row["course_name"]},
        the slug is {row["course_slug"]},
        the course technology is {row["course_technology"]},
        and the course topic is {row["course_topic"]}.
        The course description is {row["course_description"]}. 
        A short description of the course is {row["course_description_short"]}.
        A brief quote from the instructor is {row["course_instructor_quote"]}.
        The section name is {row["section_name"]} and the section description is {row["section_description"]}.
        """

In [15]:
courses['new_section_description'] = courses.apply(create_section_description, axis=1)

## Pinecone Index Creation

In [16]:
pc = Pinecone(api_key = os.environ.get('PINECONE_API_KEY'), environment = os.environ.get('PINECONE_ENV'))

In [17]:
def create_index(index_name, dimension, metric):
    if index_name in [index.name for index in pc.list_indexes()]:
        pc.delete_index(index_name)
        print(f"Deleted existing index: {index_name}")
        print(f"Creating {index_name} index with new configuration...")
        pc.create_index(name= index_name,
                        dimension=dimension,
                        metric=metric,
                        spec=ServerlessSpec(cloud="aws",
                                            region="us-east-1"))
        print(f"Created index: {index_name} with dimension: {dimension} and metric: {metric}")
    else:
        print(f"No existing index named '{index_name}' found.")
        print(f"Creating {index_name} index with new configuration...")
        pc.create_index(name= index_name,
                        dimension=dimension,
                        metric=metric,
                        spec=ServerlessSpec(cloud="aws",
                                            region="us-east-1"))
        print(f"Created index: {index_name} with dimension: {dimension} and metric: {metric}")

In [18]:
model = SentenceTransformer('all-mpnet-base-v2')

In [19]:
index_name = '365-section-descriptions'
dimension = model.get_sentence_embedding_dimension()
metric = "cosine"

create_index(index_name, dimension, metric)

No existing index named '365-section-descriptions' found.
Creating 365-section-descriptions index with new configuration...
Created index: 365-section-descriptions with dimension: 768 and metric: cosine


In [20]:
index = pc.Index(name=index_name)

## Creating Vectors and uploading to Pinecone

In [27]:
# Iterate over the dataset and prepare data for upserting
vectors_to_upsert = []
for i, row in courses.iterrows():
    
    text = row['new_section_description']
    unique_id = row['unique_id']

    # Generate the embedding for the text
    embedding = model.encode(text, show_progress_bar=False).tolist()

    metadata = {'course_name': str(row['course_name']),
                'course_slug': str(row['course_slug']),
                'course_technology': str(row['course_technology']),
                'course_topic': str(row['course_topic']),
                'course_description': str(row['course_description']),
                'course_description_short': str(row['course_description_short']),
                'course_instructor_quote': str(row['course_instructor_quote']),
                'section_name': str(row['section_name']),
                'section_description': str(row['section_description'])}

    # append the tuple, (id, embedding, metadata) to the list
    vectors_to_upsert.append((unique_id, embedding, metadata))

In [28]:
batch_size = 100
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors=batch)

print("Subset of data upserted successfully!")

Subset of data upserted successfully!


## Semantic Search

In [36]:
query = "regression in python"
query_embedding = model.encode(query, show_progress_bar=False).tolist()

In [37]:
query_results = index.query(
    vector=query_embedding,
    top_k=12,
    include_metadata=True
)

In [38]:
for q in query_results.matches:
    if q['score'] > 0.4:
        print(f"""
            Course Name: {q['metadata']['course_name']} \n
            Section Name: {q['metadata']['section_name']} \n
            Similarity Score: {q['score']} \n
            Course Description: {q['metadata']['course_description']} \n
            Section Description: {q['metadata']['section_description']} \n"""
        )


            Course Name: Machine Learning in Python 

            Section Name: Linear Regression with sklearn 

            Similarity Score: 0.571450293 

            Course Description: Machine Learning in Python builds upon the statistical knowledge you gained earlier in the program. This course focuses on predictive modelling and enters multidimensional spaces which require an understanding of mathematical methods, transformations, and distributions. We will introduce these concepts, as well as complex means of analysis such as clustering, factoring, Bayesian inference, and decision theory, while also allowing you to exercise your Python programming skills. 

            Section Description: While there are many libraries that can compute a regression model, the most numerically stable one is sklearn. It is also the preferred choice of many machine learning professionals. In this section, we implement all we know about regressions in this amazing library. 


            Course Na